In [ ]:
"""=== Part B: Proxy SVAR / SVAR-IV (Stock–Watson 2012; Mertens–Ravn 2013) ===
Same impulse responses as used in Part A, but via a structural VAR identified with an
external instrument, then propagated through VAR dynamics."""
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR                       # reduced-form VAR(4)
import statsmodels.api as sm                              # OLS for the first stage
from statsmodels.sandbox.regression.gmm import IV2SLS     # 2SLS (cross-check only)
import matplotlib.pyplot as plt

In [ ]:
"""Read the merged dataset, Data_Merge.ipynb must be run first for creating the dataset."""
df = pd.read_csv("merged_data.csv", index_col="quarter")

"""The CSV's quarter labels come back as strings; convert to a proper quarterly
datetime index so statsmodels recognises the frequency (avoids parsing warnings)."""
df.index = pd.PeriodIndex(df.index, freq="Q").to_timestamp()

In [ ]:
"""Step 1 — Reduced-form VAR(4). Order matters here: FFR is first because it is the
policy variable whose shock we identify. The VAR uses only the four macro
variables"""
VAR_variables = ["ffr", "log_real_gdp", "log_cpi", "unrate"]

""Macro series have no missing values, so this uses the FULL 1969–2019 sample,
which makes the dynamics more efficiently estimated.""
Y = df[VAR_variables].dropna()

VAR_model = VAR(Y)
var_res = VAR_model.fit(maxlags=4, trend="c")   # 4 lags + constant 

"""var_res.resid are the reduced-form residuals û_t = one-step forecast errors
("the surprises"). All identification below works on these."""
print("Sample used:", Y.index.min(), "to", Y.index.max(), "| obs:", len(Y))
print("Lag order:", var_res.k_ar)
print("Residual shape:", var_res.resid.shape)   # (200, 4) = 204 quarters - 4 lags

In [ ]:
"""Step 2 -  First stage: is the instrument relevant? Regress the FFR residual on the
external instrument z = mp_shock:   û^ffr_t = phi * z_t + eta_t."""
u_ffr = var_res.resid["ffr"]
z = df["mp_shock"].reindex(u_ffr.index)         # align shock to the residual quarters

"""Keep the quarters where both exist; the shock ends 2007Q4, so this trims to the
1970Q1–2007Q4 overlap (152 obs) where identification is possible."""
first_stage = pd.concat([u_ffr.rename("u_ffr"), z.rename("z")], axis=1).dropna()
print("First-stage sample:", first_stage.index.min(), "to", first_stage.index.max(),
      "| obs:", len(first_stage))

"""OLS with heteroskedasticity-robust (HC1) SEs; report the F-stat on z."""
fs_fit = sm.OLS(first_stage["u_ffr"], sm.add_constant(first_stage["z"])).fit(cov_type="HC1")
phi = fs_fit.params["z"]
F = fs_fit.fvalue                               # with one regressor, F = (t-stat)^2

"""Rule to follow: F > 10 => strong instrument; F < 10 => weak-instrument concern."""
print(f"\nphi (coefficient on instrument): {phi:.4f}")
print(f"First-stage F-statistic: {F:.2f}")
print("Strong instrument" if F > 10 else "WEAK instrument (F<10)")

In [ ]:
"""Step 3 — Structural impact identification. For each non-FFR variable j, 2SLS-regress
its residual on the FFR residual, Instrumented by z:  û^j_t = delta_j * û^ffr_t + ...
Instrumenting removes the simultaneity bias: it isolates the part of the
FFR surprise driven by the exogenous monetary shock. The delta_j's are the relative
impact responses; stacking with FFR normalised to 1 gives the impact vector b0."""
resid = var_res.resid.copy()
resid["z"] = df["mp_shock"].reindex(resid.index)
reg = resid.dropna()                            # overlap sample (152 obs)

u_ffr = reg["ffr"]
zz = sm.add_constant(reg["z"])                  # instrument matrix (const + z)

deltas = {"ffr": 1.0}                           # FFR normalised to 1 on impact
for j in ["log_real_gdp", "log_cpi", "unrate"]:
    iv = IV2SLS(reg[j], sm.add_constant(u_ffr), instrument=zz).fit()
    deltas[j] = iv.params["ffr"]
    """Cross-check: with one instrument, 2SLS == cov(z,u^j)/cov(z,u^ffr). They match,
    which confirms we understand the estimator (and justifies the fast form used
    in the bootstrap below)."""
    ratio = np.cov(reg["z"], reg[j])[0, 1] / np.cov(reg["z"], u_ffr)[0, 1]
    print(f"{j:14s}  delta = {iv.params['ffr']: .4f}   (cov-ratio check: {ratio: .4f})")

b0 = np.array([deltas["ffr"], deltas["log_real_gdp"],
               deltas["log_cpi"], deltas["unrate"]])
print("\nb0 =", b0)                             # [1, d_gdp, d_cpi, d_unrate]

In [ ]:
"""Step 4 — Propagate the impact vector through the VAR dynamics.
To iterate a VAR(4) forward we rewrite it as a first-order system using the
(np x np) Companion Matrix A, then IRF(h) = J A^h J' b0."""
B = var_res.coefs                               # shape (p, n, n): B[l] is lag (l+1) matrix
n = B.shape[1]                                  # number of variables = 4
p = B.shape[0]                                  # lag order = 4
print("n =", n, "| p =", p, "| B shape:", B.shape)

"""Build A: top block row holds the actual dynamics [B1 B2 B3 B4]; the identity blocks
below simply shift lags down (this quarter's value becomes next quarter's "lag 1")."""
A = np.zeros((n * p, n * p))
A[:n, :] = np.hstack([B[l] for l in range(p)])
A[n:, :-n] = np.eye(n * (p - 1))
print("Companion A shape:", A.shape)            # (16, 16)

"""J selects the current-period (top n) variables out of the stacked state vector."""
J = np.zeros((n, n * p)); J[:, :n] = np.eye(n)

"""Iterate IRF(h) = J A^h J' b0 for h = 0..20. We grow A^h incrementally for speed."""
H = 20
irf_svar = np.zeros((H + 1, n))
Ah = np.eye(n * p)                              # A^0 = identity
for h in range(H + 1):
    irf_svar[h] = J @ Ah @ J.T @ b0
    Ah = Ah @ A

irf_svar = pd.DataFrame(irf_svar, columns=VAR_variables)
irf_svar.index.name = "h"
print(irf_svar.round(4))                        # row 0 should equal b0 (sanity check)

In [ ]:
"""Scale logged variables to percent; FFR/unrate stay in their native points."""
irf_svar_pct = irf_svar.copy()
irf_svar_pct["log_real_gdp"] *= 100
irf_svar_pct["log_cpi"]      *= 100
print(irf_svar_pct[["log_real_gdp", "log_cpi"]].round(3))

In [ ]:
"""Package Steps 2–4 into two reusable functions so the bootstrap can repeat the entire estimation cleanly on each fake sample."""

def identify_b0(resid, z):
    """Structural impact vector from residuals + instrument, via the covariance
    ratio form of 2SLS:  delta_j = cov(z, u^j) / cov(z, u^ffr) on observed z.
    resid: (T x n) array with FFR in column 0; z: length-T array (may contain NaN)."""
    mask = ~np.isnan(z)                         # use only quarters where z exists
    zz = z[mask]
    u_ffr = resid[mask, 0]                      # FFR residual (column 0)
    denom = np.cov(zz, u_ffr)[0, 1]             # first-stage covariance cov(z, u^ffr)
    b0 = np.ones(resid.shape[1])               # FFR normalised to 1
    for j in range(1, resid.shape[1]):
        b0[j] = np.cov(zz, resid[mask, j])[0, 1] / denom
    return b0

def irf_from_coefs(coefs, b0, H, n, p):
    """IRF(h) = J A^h J' b0 from a VAR coefficient array (shape (p, n, n))."""
    A = np.zeros((n * p, n * p))
    A[:n, :] = np.hstack([coefs[l] for l in range(p)])   # [B1 ... Bp]
    A[n:, :-n] = np.eye(n * (p - 1))                     # shift-down identities
    J = np.zeros((n, n * p)); J[:, :n] = np.eye(n)
    out = np.zeros((H + 1, n))
    Ah = np.eye(n * p)
    for h in range(H + 1):
        out[h] = J @ Ah @ J.T @ b0
        Ah = Ah @ A
    return out

"""Sanity check: the helpers must reproduce the point estimates exactly."""
b0_check  = identify_b0(var_res.resid.values, df["mp_shock"].reindex(var_res.resid.index).values)
irf_check = irf_from_coefs(var_res.coefs, b0_check, 20, 4, 4)
print("b0 matches:", np.allclose(b0_check, b0))
print("IRF matches:", np.allclose(irf_check, irf_svar.values))

In [ ]:
"""Step 5 — Residual bootstrap for confidence bands. Treat the estimated VAR as the
true dgp, manufacture many fake histories by reshuffling residuals, re-estimate
everything on each, and read the spread of IRFs."""
rng = np.random.default_rng(42)                 # fixed seed -> reproducible
n_boot = 1000                                   # >= 500 as required
H, n, p = 20, 4, 4

"""Pre-extract dgp pieces as plain arrays (fast, and avoids date-index overhead)."""
Yv   = Y.values                                 # actual data, for initial conditions
c    = var_res.intercept                        # VAR constant
Bc   = var_res.coefs                            # lag-coefficient matrices
uhat = var_res.resid.values                     # reduced-form residuals (200 x 4)
T    = uhat.shape[0]
z_resid = df["mp_shock"].reindex(var_res.resid.index).values   # instrument (NaN post-2007)

boot_irfs = np.full((n_boot, H + 1, n), np.nan)

for b in range(n_boot):
    """(a) Resample residual ROWS with replacement. CRUCIAL: draw ONE index vector and
        apply it to BOTH residuals and instrument, so each residual stays paired
        with its own z. Shuffling them independently would destroy cov(z, u^ffr),
        on which identification depends."""
    idx    = rng.integers(0, T, size=T)
    u_star = uhat[idx]
    z_star = z_resid[idx]

    """(b) Reconstruct a fake sample recursively with the ESTIMATED coefficients,
        fixing the first p observations as initial conditions."""
    Ys = np.empty((p + T, n)); Ys[:p] = Yv[:p]
    for t in range(p, p + T):
        val = c.copy()
        for l in range(1, p + 1):
            val = val + Bc[l - 1] @ Ys[t - l]
        Ys[t] = val + u_star[t - p]

    """(c) Re-estimate the VAR on the fake data (this generates coefficient uncertainty)."""
    m_star = VAR(Ys).fit(maxlags=p, trend="c")

    """(d)+(e) Re-identify b0 and re-propagate -> one bootstrapped IRF draw."""
    boot_irfs[b] = irf_from_coefs(m_star.coefs, identify_b0(m_star.resid, z_star), H, n, p)

print("Bootstrap done. Shape:", boot_irfs.shape)   # (1000, 21, 4)
print("Any failed draws (NaN):", np.isnan(boot_irfs).any())

In [ ]:
"""Convert the 1000 bootstrap draws into pointwise confidence bands: at each horizon,
take percentiles ACROSS the draws (axis=0). 68% band = [16th, 84th] pct;
90% band = [5th, 95th] pct."""
def svar_bands(boot_irfs, point_irf, var_index, scale=1.0):
    """var_index: column in VAR order (0=ffr,1=gdp,2=cpi,3=unrate); scale=100 for logs."""
    draws = boot_irfs[:, :, var_index] * scale
    return pd.DataFrame({
        "h":    np.arange(draws.shape[1]),
        "beta": point_irf.iloc[:, var_index].values * scale,   # point estimate 
        "lo68": np.percentile(draws, 16, axis=0),
        "hi68": np.percentile(draws, 84, axis=0),
        "lo90": np.percentile(draws,  5, axis=0),
        "hi90": np.percentile(draws, 95, axis=0),
    })

"""GDP = column 1, CPI = column 2; both logged -> scale to percent."""
svar_gdp = svar_bands(boot_irfs, irf_svar, var_index=1, scale=100)
svar_cpi = svar_bands(boot_irfs, irf_svar, var_index=2, scale=100)

In [ ]:
"""Plotting IRF Functions with 68% and 90% bands"""
def plot_irf(irf_b, title, ylabel, fname):
    
    h = irf_b["h"]
    fig, ax = plt.subplots(figsize=(7, 4.5))

    # 90% band: lighter shade 
    ax.fill_between(h, irf_b["lo90"], irf_b["hi90"],
                    color="steelblue", alpha=0.20, label="90% CI")
    # 68% band: darker shade
    ax.fill_between(h, irf_b["lo68"], irf_b["hi68"],
                    color="steelblue", alpha=0.40, label="68% CI")
    # point estimate:
    ax.plot(h, irf_b["beta_pct"], color="navy", lw=2, label="Point estimate")
    # horizontal zero line
    ax.axhline(0, color="black", lw=0.8, ls="--")

    ax.set_title(title)
    ax.set_xlabel("Quarters after shock")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(0, 21, 2))
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.show()

In [ ]:
"""plot_irf function expects a column named 'beta_pct'; our SVAR tables already hold the
percent-scaled point estimate in 'beta', so it is copied across"""
for tab in (svar_gdp, svar_cpi):
    tab["beta_pct"] = tab["beta"]

plot_irf(svar_gdp,
         title="Response of Real GDP to a Monetary Policy Shock (Proxy SVAR)",
         ylabel="Real GDP response (% deviation from baseline)",
         fname="irf_gdp_svar.png")

plot_irf(svar_cpi,
         title="Response of CPI to a Monetary Policy Shock (Proxy SVAR)",
         ylabel="CPI response (% deviation from baseline)",
         fname="irf_cpi_svar.png")